# Week 1 — live-coding notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week01/week1_demos.ipynb)

At the first break, run **Setup** (when present) and **Imports**. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. For a complete preflight check, use **Runtime → Run all**.

Each task follows the same rhythm: predict, run, change one declared input, and record the requested artifact. The notebook is self-contained, so you can run it in Colab or in your local course repository.

## Cumulative research record

Use this Markdown section to practice the research-record habit. Keep these entries in this weekly notebook; Weeks 1–5 practice records are not merged or submitted separately. When your project begins in Week 6, start one project-root `RESEARCH_RECORD.md` and maintain that file through the final submission.

### Mini-project 1 — Learning brief

After running Demo 1, choose **option pricing**, **claim frequency**, or **daily return direction**. Replace each prompt below with one sentence. You are planning a fair learning experiment, not fitting a model yet.

1. **Target:** I want to estimate `[outcome]`, measured in `[units]`.
2. **Source of structure:** This may be learnable because `[economic or statistical reason]`.
3. **Evidence:** On later observations, I would compare `[proposed approach]` with `[simplest credible baseline]` using `[loss or score]`.
4. **Decision:** If the proposed approach improves that comparison, `[person or system]` would change `[specific action]`.

Do not report a result in this entry. The brief states what evidence would count before the result is visible.

### Mini-project 2 — Four-strategy audit

Before running Demo 2, use only the development-period chart shown in the lecture. Then replace the prompts below after the procedure and later period are revealed.

- **Development ranking:** `[rank A–D and give the rule you used]`
- **Later-period ranking:** `[rank A–D after the boundary]`
- **Evidence that changed the decision:** `[one specific procedure or result]`
- **Further test before funding:** `[one additional piece of evidence]`
- **Narrow claim:** `[what this one later period warrants—and what it does not]`

### Later Week 1 entries

After each later task, add only the fields that have become meaningful:

2. **Audit:** candidates compared, development evidence, later period, the claim the memo can warrant.
3. **Address:** data source and vintage, sample dates, seed, package versions, commit hash.
4. **Transform:** object and units, transformation applied, sample span, what it did not fix.
5. **Memory:** series and transformation, lag range, observed ACF pattern, closest classical vocabulary, and one claim the ACF cannot establish.
6. **Object:** question and decision, statistical object, feasible baseline, the loss that would settle it.

For every result, write its limitation and the narrow claim the evidence supports.

### Imports

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import acf, adfuller, coint

rng = np.random.default_rng(42)


def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url

### Demo 1 — A neural network learns an option-pricing curve · Deck A · run at the break after S1

> **First demonstration:** Deck A, after recording segment S1. Run the cell exactly as written to reproduce the option-pricing figure. You are not expected to understand every line yet; read the graph, then complete the learning brief in the Markdown cell above.

In [ ]:
# Just run this first example as written. We will unpack the machinery later.
def black_scholes_call(moneyness, maturity=1.0, volatility=0.2, rate=0.0):
    """Black–Scholes call price divided by strike, as a function of S/K."""
    m = np.asarray(moneyness)
    d1 = (np.log(m) + (rate + 0.5 * volatility**2) * maturity) / (
        volatility * np.sqrt(maturity)
    )
    d2 = d1 - volatility * np.sqrt(maturity)
    return m * norm.cdf(d1) - np.exp(-rate * maturity) * norm.cdf(d2)


option_rng = np.random.default_rng(1994)
train_x = np.sort(option_rng.uniform(0.68, 1.32, 220))
train_y = black_scholes_call(train_x)

# The network receives examples, not the Black–Scholes formula.
option_network = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 32),
        activation="tanh",
        solver="lbfgs",
        alpha=1e-7,
        max_iter=5000,
        random_state=1994,
    ),
)
option_network.fit(train_x[:, None], train_y)

grid = np.linspace(0.55, 1.45, 500)
truth = black_scholes_call(grid)
learned = option_network.predict(grid[:, None])
inside = (grid >= 0.68) & (grid <= 1.32)
abs_error = np.abs(learned - truth) * 10_000

fig, (curve, error) = plt.subplots(
    1, 2, figsize=(14, 4.7),
    gridspec_kw={"width_ratios": [3.4, 1.25], "wspace": 0.28},
)
curve.axvspan(0.68, 1.32, color="tab:blue", alpha=0.07,
              label="training domain")
curve.scatter(train_x[::5], train_y[::5], s=22, color="tab:orange",
              alpha=0.65, label="examples", zorder=2)
curve.plot(grid, truth, color="black", ls="--", lw=2.3,
           label="Black–Scholes")
curve.plot(grid, learned, color="tab:green", lw=2.7,
           label="neural network")
curve.set(xlabel="moneyness  $S/K$", ylabel="normalized call price  $C/K$",
          title="Pricing function")
curve.legend(frameon=False, ncol=2, loc="upper left")
curve.spines[["top", "right"]].set_visible(False)

error.axvspan(0.68, 1.32, color="tab:blue", alpha=0.07)
error.plot(grid, abs_error, color="tab:red", lw=2)
error.fill_between(grid[inside], 0, abs_error[inside],
                   color="tab:red", alpha=0.18)
error.set(xlabel="moneyness  $S/K$", ylabel="basis points of strike",
          title="Absolute error")
error.spines[["top", "right"]].set_visible(False)

fig.suptitle("Examples teach a network the Black–Scholes pricing curve",
             fontweight="bold", fontsize=17)
plt.show()

### Demo 2 — Audit four strategies across a frozen boundary · Deck A · run at the break after S2

> **Audit demonstration:** Deck A, after recording segment S2. Before running the cell, rank A–D using only the development-period chart in the lecture. Then run the cell to reveal the procedures and later-period evidence, and complete the four-strategy audit in the Markdown cell above.

In [ ]:
# Before running: rank A–D using only the development-period chart in the lecture.
# These annual observations are a compact, self-contained extract from the
# chapter experiment. Students do not need the chapter notebook or its assets.
audit_csv = """Date,A development,B development,C development,D development,equal-risk development,60/40 development,A later,B later,C later,D later,equal-risk later,60/40 later
1990-12-31,0.096331,0.166288,0.025916,0.040344,0.026749,-0.018902,,,,,,
1991-12-31,0.270206,0.398369,0.221376,0.223311,0.222209,0.185371,,,,,,
1992-12-31,0.310507,0.542604,0.272951,0.274886,0.273784,0.231518,,,,,,
1993-12-31,0.386650,0.693849,0.376993,0.378927,0.377826,0.325896,,,,,,
1994-12-31,0.431311,0.813902,0.322752,0.307956,0.323585,0.275938,,,,,,
1995-12-31,0.590074,1.066986,0.586694,0.519213,0.587527,0.547363,,,,,,
1996-12-31,0.631774,1.237940,0.650335,0.558830,0.653472,0.657060,,,,,,
1997-12-31,0.723392,1.524769,0.804550,0.702966,0.806298,0.867731,,,,,,
1998-12-31,0.919045,1.792613,0.981374,0.877850,0.983122,1.065427,,,,,,
1999-12-31,0.993572,1.904541,0.965881,0.862281,0.967629,1.164899,,,,,,
2000-12-31,1.192106,2.100887,1.063914,0.926406,1.065662,1.134903,,,,,,
2001-12-31,1.331326,2.378850,1.063601,0.960968,1.060542,1.060145,,,,,,
2002-12-31,1.548368,2.693418,1.100121,1.041252,1.094866,0.957544,,,,,,
2003-12-31,1.678232,2.992079,1.171960,1.069607,1.171483,1.092987,,,,,,
2004-12-31,1.729482,3.045472,1.222027,1.106493,1.222508,1.162173,,,,,,
2005-12-31,1.747063,3.064063,1.245984,1.117567,1.247595,1.193522,,,,,,
2006-12-31,1.766374,3.111119,1.299186,1.138477,1.300797,1.281186,,,,,,
2007-12-31,1.905863,3.164838,1.372335,1.211626,1.373946,1.338262,,,,,,
2008-12-31,2.239120,3.600839,1.408529,1.340622,1.409775,1.151128,,,,,,
2009-12-31,2.501040,4.159480,1.398912,1.303651,1.415757,1.209716,,,,,,
2010-12-31,2.620048,4.353163,1.494437,1.368384,1.511281,1.306572,,,,,,
2011-12-31,2.778311,4.659325,1.593648,1.449243,1.610493,1.368317,,,,,,
2012-12-31,2.855879,4.699517,1.636808,1.488756,1.652203,1.446510,,,,,,
2013-12-31,2.876811,4.753885,1.651963,1.554208,1.674930,1.566648,,,,,,
2014-12-31,2.901256,4.782360,1.740670,1.605164,1.763637,1.667241,,,,,,
2015-12-31,2.986344,4.826528,1.730397,1.583812,1.757009,1.667174,,,,,,
2016-12-31,,,,,,,-0.071805,-0.008148,0.017358,-0.007848,0.025960,0.057876
2017-12-31,,,,,,,-0.094435,0.015968,0.099162,0.059565,0.107764,0.180430
2018-12-31,,,,,,,-0.190159,0.011526,0.102145,0.048754,0.109916,0.141186
2019-12-31,,,,,,,-0.223667,0.060602,0.213997,0.146344,0.225395,0.334958
2020-12-31,,,,,,,-0.303455,0.035772,0.278480,0.185249,0.299239,0.464004
2021-12-31,,,,,,,-0.369369,0.026421,0.314014,0.214940,0.334773,0.627149
2022-12-31,,,,,,,-0.597438,-0.095679,0.133315,0.145143,0.167337,0.427354
2023-12-31,,,,,,,-0.630700,-0.080795,0.241021,0.189581,0.276502,0.599762
2024-12-31,,,,,,,-0.707835,-0.110017,0.306231,0.218679,0.341712,0.762172
2025-12-31,,,,,,,-0.814853,-0.136956,0.375093,0.274179,0.421664,0.898920
2026-12-31,,,,,,,-0.840808,-0.174212,0.390377,0.295696,0.443181,0.973976
"""
audit = pd.read_csv(StringIO(audit_csv), parse_dates=["Date"]).set_index("Date")

colors = {"A": "tab:red", "B": "tab:orange", "C": "tab:blue", "D": "tab:green"}
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for key, color in colors.items():
    axes[0].plot(audit.index, audit[f"{key} development"], label=key, color=color, lw=2)
    axes[1].plot(audit.index, audit[f"{key} later"], label=key, color=color, lw=2)
for ax, suffix in zip(axes, ("development", "later")):
    ax.plot(audit.index, audit[f"equal-risk {suffix}"], color="0.35", ls="--", label="equal-risk")
    ax.plot(audit.index, audit[f"60/40 {suffix}"], color="0.7", ls=":", label="60/40")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_ylabel("cumulative log return")
    ax.legend(frameon=False, ncol=2, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_title("1. Development period: the tempting ranking")
axes[1].set_title("2. Frozen later period: the ranking changes")
fig.tight_layout()
plt.show()

audit_key = pd.DataFrame(
    {
        "model": [
            "degree-10 polynomial OLS",
            "degree-10 polynomial OLS",
            "Lasso",
            "fixed 12-month momentum rule",
        ],
        "features": [
            "four noise series",
            "four real momentum windows",
            "four real momentum windows",
            "one 12-month momentum signal",
        ],
        "selection / validation": [
            "none",
            "none",
            "walk-forward; expanding window",
            "rule fixed in advance",
        ],
    },
    index=["A", "B", "C", "D"],
)
print(audit_key.to_string())
print("\nStored values are year-end extracts from the course's full daily backtest.")
print("The later period can reject a procedure here; it cannot establish a universal winner.")

### Demo 3 — A random walk and its returns side by side · Deck B · run at the break after S4

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, run the cell, then complete the requested comparison and record the requested artifact.

In [ ]:
T = 1000
steps = rng.standard_normal(T) * 0.01
prices = 100 * np.exp(np.cumsum(steps))
simple_returns = np.diff(prices) / prices[:-1]
returns = np.diff(np.log(prices))

print(f"Maximum |simple - log return|: {np.max(np.abs(simple_returns - returns)):.2e}")
print(f"Cumulative simple return: {prices[-1] / prices[0] - 1:+.2%}")
print(f"Sum of log returns:       {returns.sum():+.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(prices, lw=0.8)
axes[0].set_title("Price (non-stationary — drifts without bound)")
axes[0].set_xlabel("t"); axes[0].set_ylabel("Price")
axes[1].plot(returns, lw=0.5, color="tab:red")
axes[1].plot(simple_returns, lw=0.4, color="tab:blue", alpha=0.6,
             label="simple return")
axes[1].set_title("Log return (stationary — bounded scale)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("Log return")
axes[1].legend()
fig.tight_layout()
plt.show()
# Expected: the two conventions agree to ~1e-3 on daily moves, yet the totals differ:
#           log returns sum (-0.29) while simple returns compound (-25%).

### Real-data companion — US market wealth, returns, and tail days · Deck B · optional after S4

> **Optional empirical comparison:** Deck B, after recording segment S4. Run this after the known-truth demo. Keep the procedure fixed, then compare the sign, scale, stability, and limitations of the historical result. A disagreement is evidence to explain, not a reason to retune the simulation.

In [ ]:
market_raw, market_source = course_csv("datasets/famafrench/ff_factors_daily.csv")
market_raw["date"] = pd.to_datetime(market_raw["date"].astype(str), format="%Y%m%d")
market_data = market_raw.set_index("date").sort_index()
market_daily_return = (market_data["Mkt-RF"] + market_data["RF"]) / 100.0
market_wealth = (1 + market_daily_return).cumprod()
market_recent = market_daily_return.loc["2000":]

print("Source: Kenneth R. French Data Library, bundled daily US market factor")
print(f"File: {market_source}")
print(f"Sample: {market_daily_return.index.min().date()} through {market_daily_return.index.max().date()}")
print("Units: growth of $1 for the level; decimal simple return for the change")
print(f"2000+ annualized volatility: {market_recent.std() * np.sqrt(252):.1%}")
print(f"2000+ share beyond 3 sample SD: {(market_recent.abs() > 3 * market_recent.std()).mean():.2%}")
print("Largest absolute daily moves since 2000:")
print(market_recent.loc[market_recent.abs().nlargest(5).index].sort_index().map("{:+.2%}".format))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(market_wealth.loc["2000":], lw=0.8)
axes[0].set_yscale("log")
axes[0].set_title("US market wealth index (log scale)")
axes[0].set_ylabel("Growth of $1 since 1926")
axes[1].plot(market_recent, lw=0.45, color="tab:red")
axes[1].axhline(0, color="0.3", lw=0.5)
axes[1].set_title("Daily simple returns: bounded, but not tame")
axes[1].set_ylabel("Simple return")
fig.tight_layout(); plt.show()
print("One history cannot identify a data-generating process; it can show which realized patterns need explanation.")

### Demo 4 — Read memory from the ACF · Deck B · run at the break after S5

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, run the cell, then complete the requested comparison and record the requested artifact.

In [ ]:
T_memory = 700
memory_rng = np.random.default_rng(7057)

ar_shocks = memory_rng.standard_normal(T_memory)
ar_series = np.zeros(T_memory)
for t in range(1, T_memory):
    ar_series[t] = 0.72 * ar_series[t - 1] + ar_shocks[t]

ma_shocks = memory_rng.standard_normal(T_memory + 1)
ma_series = ma_shocks[1:] + 0.72 * ma_shocks[:-1]

memory_series = {
    "AR(1): past values echo": ar_series,
    "MA(1): one old shock lingers": ma_series,
    "US market return": market_recent.to_numpy(),
    "Squared US market return": market_recent.to_numpy() ** 2,
}

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharex=True)
for ax, (name, values) in zip(axes.flat, memory_series.items()):
    rho = acf(values, nlags=21, fft=True)
    markerline, stemlines, baseline = ax.stem(range(1, 22), rho[1:])
    plt.setp(markerline, markersize=3.5)
    plt.setp(stemlines, linewidth=1.2)
    baseline.set_color("0.65")
    bound = 1.96 / np.sqrt(len(values))
    ax.axhline(bound, color="tab:red", ls="--", lw=0.8)
    ax.axhline(-bound, color="tab:red", ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_ylabel("autocorrelation")
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("lag")
fig.suptitle("Different memories leave different ACF fingerprints", fontweight="bold")
fig.tight_layout()
plt.show()

print("AR(1): the population ACF fades geometrically across lags.")
print("MA(1): the population ACF is zero after lag 1; sample bars still wobble.")
print("Market: compare weak signed-return memory with persistent magnitude memory.")


def adf_summary(name, x, regression="c", autolag="AIC"):
    stat, p_value, selected_lag, nobs, *_ = adfuller(
        x, regression=regression, autolag=autolag
    )
    conclusion = (
        "reject the specified unit-root null"
        if p_value < 0.05
        else "fail to reject the specified unit-root null"
    )
    print(
        f"{name}: N={len(x)}, regression='{regression}', autolag='{autolag}', "
        f"selected lag={selected_lag}, ADF statistic={stat:+.3f}, "
        f"p-value={p_value:.4f} -> {conclusion}"
    )
    return {
        "series": name,
        "n": len(x),
        "regression": regression,
        "autolag": autolag,
        "selected_lag": selected_lag,
        "nobs_used": nobs,
        "statistic": stat,
        "p_value": p_value,
        "conclusion": conclusion,
    }

adf_log_price = adf_summary("log price", np.log(prices))
adf_log_return = adf_summary("log return", returns)
# Expected under this known-truth construction: fail to reject for the log
# price and reject for the log return. Neither output certifies a permanent
# description of an unknown real-world process.

### Demo 5 — Spurious regression on two independent random walks · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, run the cell, then complete the requested comparison and record the requested artifact.

In [ ]:
T = 500
n_trials = 200
big_t_stats = 0
for _ in range(n_trials):
    a = np.cumsum(rng.standard_normal(T))
    b = np.cumsum(rng.standard_normal(T))
    # Simple OLS of b on a
    A = np.column_stack([np.ones(T), a])
    beta, *_ = np.linalg.lstsq(A, b, rcond=None)
    resid = b - A @ beta
    sigma2 = resid @ resid / (T - 2)
    se = np.sqrt(sigma2 * np.linalg.inv(A.T @ A)[1, 1])
    t = beta[1] / se
    if abs(t) > 2:
        big_t_stats += 1

print(f"Of {n_trials} independent random-walk pairs, "
      f"{big_t_stats} had |t| > 2 on the slope coefficient.")
# Expected: well over the 5% that classical OLS would predict — the t-stat
# distribution is *broken* under unit-root regressors.

### Demo 6 — A cointegrated pair, and its stationary spread · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, run the cell, then complete the requested comparison and record the requested artifact.

In [ ]:
T = 1000
shared = np.cumsum(rng.standard_normal(T))          # common stochastic trend
x = shared + rng.standard_normal(T) * 0.5
y = 0.8 * shared + rng.standard_normal(T) * 0.5     # noisy linear function
spread = y - 0.8 * x

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(x, label="x", lw=0.8)
axes[0].plot(y, label="y", lw=0.8)
axes[0].set_title("Two cointegrated series (both non-stationary)")
axes[0].legend()
axes[1].plot(spread, color="tab:purple", lw=0.6)
axes[1].set_title("Their spread y − 0.8x (stationary)")
fig.tight_layout()
plt.show()

adf_summary("x", x)
adf_summary("y", y)
adf_summary("spread", spread)
# Spread should be stationary (low p); the levels are not.

stat, p, _ = coint(x, y)
print(f"Engle-Granger cointegration test: stat = {stat:+.3f}, p = {p:.4f}")
# Expected: x and y each fail to reject a unit root; their spread rejects decisively
#           and Engle-Granger rejects. Cointegration belongs to the pair, not to either series.

### Demo 7 — Multiple-testing teaser (connects to Part A and Ch 16) · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, run the cell, then complete the requested comparison and record the requested artifact.

In [ ]:
# Generate 1000 random strategies on independent IID noise; report the
# best-Sharpe one. This is the basis for the "deflated Sharpe" intuition
# we'll come back to in Ch 16.
T = 252 * 3  # 3 years of daily returns
n_strats = 1000
strats = rng.standard_normal((n_strats, T)) * 0.01

sharpes = strats.mean(axis=1) / strats.std(axis=1) * np.sqrt(252)
print(f"True Sharpe of each strategy: 0 (IID noise, no skill).")
print(f"Mean Sharpe across strats:  {sharpes.mean():+.3f}")
print(f"Best Sharpe among {n_strats}:    {sharpes.max():+.3f}")
print(f"Worst Sharpe:                  {sharpes.min():+.3f}")
# The 'best' one is impressive — and pure luck.
# Expected: true Sharpe is 0 for every strategy. The mean lands near 0, the best of
#           1,000 near +1.7. That number is search, not skill.